# Finance Analytics Dashboard — Basic to Intermediate



## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

pd.set_option("display.max_columns", None)
print("Ready to build the finance dashboard!")

Ready to build the finance dashboard!


## 2. Create sample finance data

In [2]:
np.random.seed(42)
months = pd.date_range("2025-01-01", periods=12, freq="MS")
departments = ["Technology", "Marketing", "Operations", "Human Resources", "Finance"]
records = []

for month in months:
    for department in departments:
        revenue = np.random.randint(70000, 180000)
        expenses = np.random.randint(35000, 125000)
        budget = np.random.randint(60000, 115000)
        records.append({
            "Month": month, "Department": department,
            "Revenue": revenue, "Expenses": expenses, "Budget": budget
        })

df = pd.DataFrame(records)
display(df.head())
print("Rows and columns:", df.shape)

,Month,Department,Revenue,Expenses,Budget
0,2025-01-01,Technology,85795,35860,98158
1,2025-01-01,Marketing,146820,89886,66265
2,2025-01-01,Operations,152386,72194,81962
3,2025-01-01,Human Resources,114131,95263,76023
4,2025-01-01,Finance,111090,102221,60769


Rows and columns: (60, 5)


## 3. Explore and clean data

In [3]:
print("Data types:")
display(df.dtypes.to_frame("Data type"))
print("Missing values:")
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())

df["Month"] = pd.to_datetime(df["Month"], errors="coerce")
for col in ["Revenue", "Expenses", "Budget"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.drop_duplicates().dropna(subset=["Month", "Department", "Revenue", "Expenses", "Budget"]).copy()
print("Cleaned rows:", len(df))

Data types:


,Data type
Month,datetime64[ns]
Department,object
Revenue,int64
Expenses,int64
Budget,int64


Missing values:


,Missing values
Month,0
Department,0
Revenue,0
Expenses,0
Budget,0


Duplicate rows: 0
Cleaned rows: 60


## 4. Create calculated columns

In [4]:
df["Profit"] = df["Revenue"] - df["Expenses"]
df["Profit_Margin_%"] = np.where(df["Revenue"] != 0, df["Profit"] / df["Revenue"] * 100, 0)
df["Budget_Variance"] = df["Expenses"] - df["Budget"]
df["Budget_Status"] = np.where(df["Expenses"] > df["Budget"], "Over Budget", "Within Budget")
display(df.head())

,Month,Department,Revenue,Expenses,Budget,Profit,Profit_Margin_%,Budget_Variance,Budget_Status
0,2025-01-01,Technology,85795,35860,98158,49935,58.202692,-62298,Within Budget
1,2025-01-01,Marketing,146820,89886,66265,56934,38.778096,23621,Over Budget
2,2025-01-01,Operations,152386,72194,81962,80192,52.624257,-9768,Within Budget
3,2025-01-01,Human Resources,114131,95263,76023,18868,16.531880,19240,Over Budget
4,2025-01-01,Finance,111090,102221,60769,8869,7.983617,41452,Over Budget


## 5. Financial KPI cards

In [5]:
total_revenue = df["Revenue"].sum()
total_expenses = df["Expenses"].sum()
total_profit = df["Profit"].sum()
overall_margin = total_profit / total_revenue * 100 if total_revenue else 0
over_budget = (df["Expenses"] > df["Budget"]).sum()

print(f"TOTAL REVENUE: ₹{total_revenue:,.0f}")
print(f"TOTAL EXPENSES: ₹{total_expenses:,.0f}")
print(f"TOTAL PROFIT: ₹{total_profit:,.0f}")
print(f"OVERALL PROFIT MARGIN: {overall_margin:.2f}%")
print(f"Department-month records over budget: {over_budget}")

TOTAL REVENUE: ₹7,360,289
TOTAL EXPENSES: ₹4,804,832
TOTAL PROFIT: ₹2,555,457
OVERALL PROFIT MARGIN: 34.72%
Department-month records over budget: 28


## 6. Monthly financial trends

In [6]:
monthly = df.groupby("Month", as_index=False)[["Revenue", "Expenses", "Profit", "Budget"]].sum()
monthly["Revenue_3M_Avg"] = monthly["Revenue"].rolling(3).mean()

fig = px.line(monthly, x="Month", y=["Revenue", "Expenses", "Profit"],
              markers=True, title="Monthly Revenue, Expenses and Profit")
fig.show()
display(monthly)

,Month,Revenue,Expenses,Profit,Budget,Revenue_3M_Avg
0,2025-01-01,610222,395424,214798,383177,NaN
1,2025-02-01,611798,378157,233641,381667,NaN
2,2025-03-01,617568,407972,209596,400000,613196.000000
3,2025-04-01,657424,468619,188805,402635,628930.000000
4,2025-05-01,666895,386919,279976,411469,647295.666667
5,2025-06-01,563909,402458,161451,401170,629409.333333
6,2025-07-01,586909,348380,238529,416457,605904.333333
7,2025-08-01,537709,348944,188765,448023,562842.333333
8,2025-09-01,596928,556946,39982,436753,573848.666667
9,2025-10-01,635753,353982,281771,465395,590130.000000


## 7. Department performance

In [7]:
department_summary = df.groupby("Department", as_index=False)[["Revenue", "Expenses", "Profit", "Budget"]].sum()
department_summary["Profit_Margin_%"] = (department_summary["Profit"] / department_summary["Revenue"] * 100).round(2)

fig = px.bar(department_summary.sort_values("Profit", ascending=False),
             x="Department", y="Profit", color="Profit",
             title="Total Profit by Department")
fig.show()
display(department_summary.sort_values("Profit", ascending=False))

,Department,Revenue,Expenses,Profit,Budget,Profit_Margin_%
3,Operations,1445319,782614,662705,980296,45.85
4,Technology,1578102,1008297,569805,1034345,36.11
1,Human Resources,1557102,1009188,547914,1018596,35.19
0,Finance,1368506,954796,413710,911199,30.23
2,Marketing,1411260,1049937,361323,1039366,25.60


## 8. Budget vs actual expenses

In [8]:
budget_summary = df.groupby("Department", as_index=False)[["Expenses", "Budget"]].sum()
budget_summary["Variance"] = budget_summary["Expenses"] - budget_summary["Budget"]
budget_summary["Status"] = np.where(budget_summary["Variance"] > 0, "Over Budget", "Within Budget")

fig = px.bar(budget_summary, x="Department", y=["Expenses", "Budget"],
             barmode="group", title="Actual Expenses vs Budget")
fig.show()
display(budget_summary)

,Department,Expenses,Budget,Variance,Status
0,Finance,954796,911199,43597,Over Budget
1,Human Resources,1009188,1018596,-9408,Within Budget
2,Marketing,1049937,1039366,10571,Over Budget
3,Operations,782614,980296,-197682,Within Budget
4,Technology,1008297,1034345,-26048,Within Budget


## 9. Interactive department filter

## 10. Export analysed data

In [10]:
df.to_csv("finance_data_analysed.csv", index=False)
department_summary.to_csv("department_finance_summary.csv", index=False)
print("Saved finance_data_analysed.csv and department_finance_summary.csv")

# In Google Colab, uncomment these lines if you want to download the files:
# from google.colab import files
# files.download("finance_data_analysed.csv")
# files.download("department_finance_summary.csv")

Saved finance_data_analysed.csv and department_finance_summary.csv
